# Workbook Overview
### Aims
Engineer the features that will be used within the model within this notebook which can then be transferred into a .py file in the src area of the repo

### Factors which will be Engineered
1. Temporal features (DOW)
2. Lag Features (shift recovery score to align with yesterdays strain etc)
3. Transforming & Normalising metrics
5. Feature Selection
6. Train Test Split the data ready for modelling

# 1. Environment Setup

In [1]:
import pandas as pd 
import numpy as np 
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import seaborn as sns
import matplotlib.pyplot as plt 
import time
from dotenv import load_dotenv
import os
import math



# 2.1 Database Connection

In [2]:
load_dotenv(dotenv_path="../.env")

USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DBNAME = os.getenv("DB_NAME")

# Construct the SQLAlchemy connection string
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!


# 2.2 Data Retrieval

In [3]:

query = text(
    """SELECT 
        -- identifiers
        R.cycle_id
        , R.created_at::date  AS date

        -- recovery score
        , R.recovery_score

        -- physiological signals
        , R.resting_heart_rate
        , R.hrv_rmssd_milli
        , R.spo2_percentage
        , R.skin_temp_celsius

        -- sleep timing
        , S.start AS sleep_start
        , S.end AS sleep_end

        -- raw sleep stage millis (convert to hours in Python)
        , S.total_in_bed_time_milli
        , S.total_awake_time_milli
        , S.total_light_sleep_time_milli
        , S.total_slow_wave_sleep_time_milli
        , S.total_rem_sleep_time_milli

        -- sleep quality metrics
        , S.sleep_efficiency_percentage
        , S.sleep_consistency_percentage
        , S.sleep_performance_percentage
        , S.respiratory_rate
        , S.sleep_cycle_count
        , S.disturbance_count

        -- sleep debt
        , S.sleep_needed_baseline_milli
        , S.sleep_needed_need_from_recent_strain_milli

        -- daily strain
        , C.strain AS cycle_strain
        , C.average_heart_rate AS cycle_avg_heart_rate
        , C.max_heart_rate AS cycle_max_heart_rate
        , C.kilojoule AS cycle_kilojoule

    FROM 
        fact_recovery R
    LEFT JOIN 
        fact_activity_sleep S 
        ON R.sleep_id = S.sleep_id
    LEFT JOIN 
        fact_cycle C 
        ON R.cycle_id = C.cycle_id

    WHERE 
        1=1
        AND S.nap = FALSE
        AND S.total_no_data_time_milli < 0.1

    ORDER BY R.created_at ASC
        """)

df = pd.read_sql_query(query, con=engine)
df.head()

,cycle_id,date,recovery_score,resting_heart_rate,hrv_rmssd_milli,spo2_percentage,skin_temp_celsius,sleep_start,sleep_end,total_in_bed_time_milli,...,sleep_performance_percentage,respiratory_rate,sleep_cycle_count,disturbance_count,sleep_needed_baseline_milli,sleep_needed_need_from_recent_strain_milli,cycle_strain,cycle_avg_heart_rate,cycle_max_heart_rate,cycle_kilojoule
0,485128018,2024-01-01,39,63,55.290910,96.15385,35.300000,2024-01-01 01:29:28.064,2024-01-01 06:25:33.135,17764110,...,54.0,13.437500,3,8,27915179,534836,14.368253,78,186,11146.555
1,485691691,2024-01-02,63,60,63.988388,96.50000,NaN,2024-01-01 23:23:56.167,2024-01-02 05:29:42.835,21945707,...,61.0,13.828125,4,11,27915179,1960856,12.028449,79,185,10150.765
2,486293849,2024-01-03,52,62,55.371407,94.79412,34.600000,2024-01-02 22:53:21.257,2024-01-03 05:37:35.250,24253031,...,65.0,13.857422,5,9,27914873,1190903,5.601850,72,143,8887.534
3,486924019,2024-01-04,59,60,59.133987,95.75000,35.100000,2024-01-03 23:30:49.228,2024-01-04 07:34:50.589,29040400,...,78.0,13.535156,6,16,27914568,226990,12.666767,75,179,10591.021
4,487510229,2024-01-05,73,60,64.707344,95.37500,35.113335,2024-01-04 22:39:14.320,2024-01-05 07:29:53.066,31318679,...,83.0,13.525391,8,17,27914263,1375608,4.392954,69,130,8900.495


# 3.1 Data Wrangling
Conversion of millisecond columns into minute columns for improved interprability

Column selection for the dataset

In [13]:
# Convert the millisecond columns into minutes for bettet interpretability and visualization in Python. 

df_wrangled = df.copy()
milli_cols = ['total_in_bed_time_milli',
               'total_awake_time_milli',
               'total_light_sleep_time_milli',
               'total_slow_wave_sleep_time_milli',
               'total_rem_sleep_time_milli',
               'sleep_needed_need_from_recent_strain_milli',
               'sleep_needed_baseline_milli'
               
               ]
for col in milli_cols:
    df_wrangled[col.replace('_milli', '_hours')] = df_wrangled[col] / 3600000

df_wrangled =df_wrangled.drop(columns=milli_cols)

df_wrangled['total_sleep_time_hours'] = df_wrangled['total_light_sleep_time_hours'] 
+ df_wrangled['total_slow_wave_sleep_time_hours'] 
+ df_wrangled['total_rem_sleep_time_hours']


cols = [
    # identifiers
    'cycle_id',
    'date',

    # recovery score
    'recovery_score',
    
    # physiological metrics
    'hrv_rmssd_milli',
    'resting_heart_rate',
    'spo2_percentage',
    'skin_temp_celsius',
    'respiratory_rate',
    
    # sleep duration
    'total_in_bed_time_hours',
    'total_awake_time_hours',
    
    # sleep stages (minutes)
    'total_light_sleep_time_hours',
    'total_slow_wave_sleep_time_hours',
    'total_rem_sleep_time_hours',
    'total_sleep_time_hours',
    
    # sleep quality
    'sleep_efficiency_percentage',
    'sleep_consistency_percentage',
    'sleep_performance_percentage',
    'sleep_cycle_count',
    'disturbance_count',
    
    # sleep timing
    'sleep_start',
    'sleep_end',
    
    # sleep debt
    'sleep_needed_baseline_hours',
    'sleep_needed_need_from_recent_strain_hours',
    
    # strain
    'cycle_strain',
    'cycle_avg_heart_rate',
    'cycle_max_heart_rate',
    'cycle_kilojoule',
]

df_wrangled = df_wrangled[cols]

df_wrangled.head()
df_wrangled.describe().T

,count,mean,min,25%,50%,75%,max,std
cycle_id,830.0,860449880.626506,485128018.0,624350278.25,812161227.0,1073064303.5,1437064952.0,269179245.919201
recovery_score,830.0,60.304819,1.0,47.0,62.0,75.0,98.0,20.296294
hrv_rmssd_milli,830.0,66.328122,16.728266,58.351706,67.376565,75.552805,105.77171,14.097413
resting_heart_rate,830.0,59.415663,51.0,56.0,58.0,61.0,86.0,5.138093
spo2_percentage,828.0,96.283414,92.57143,95.712187,96.333336,96.941995,98.85714,0.989455
skin_temp_celsius,820.0,34.954435,33.105,34.757999,35.0,35.2,35.9,0.347853
respiratory_rate,830.0,13.522832,12.30957,13.242188,13.476562,13.710938,16.289062,0.443625
total_in_bed_time_hours,830.0,7.369268,3.103082,6.480587,7.241685,8.177651,11.984278,1.239028
total_awake_time_hours,830.0,0.675484,0.058333,0.456622,0.617515,0.823154,2.656838,0.326936
total_light_sleep_time_hours,830.0,3.044092,1.265475,2.403272,2.974572,3.570663,6.316162,0.835634


In [ ]:
## The above describe finds 2 values are missing for spo2_percentage and skin_temp_celsius.

print(f"Nulls in df_wrangled: {df_wrangled['spo2_percentage'].isnull().sum()}")

# Find which rows are null
df_wrangled[df_wrangled['spo2_percentage'].isnull()][['date', 'recovery_score', 'spo2_percentage']]

# Infill with the median for these 2 valus

df_wrangled['spo2_percentage'] = df_wrangled['spo2_percentage'].fillna(df_wrangled['spo2_percentage'].median())

## Skin temp has 10 missing values

In [20]:
## Skin temp has 10 missing values, we will handle this in the same way
# Find which rows are null

print(f"Nulls in df_wrangled: {df_wrangled['skin_temp_celsius'].isnull().sum()}")

df_wrangled[df_wrangled['skin_temp_celsius'].isnull()][['date', 'recovery_score', 'skin_temp_celsius']]

df_wrangled['skin_temp_celsius'] = df_wrangled['skin_temp_celsius'].fillna(df_wrangled['skin_temp_celsius'].median())

Nulls in df_wrangled: 10


# 3.2 Removal of columns from 01_eda notebook

In [6]:
df_wrangled = df_wrangled.drop(columns=['total_light_sleep_time_hours']) # Perfect colinearity with total sleep

df_wrangled['date'] = pd.to_datetime(df_wrangled['date']) # Converts date to datetime data type instead of object



In [7]:
print(f"Shape: {df_wrangled.shape}")
print(f"Date range: {df_wrangled['date'].min()} to {df_wrangled['date'].max()}")
print(f"Date dtype: {df_wrangled['date'].dtype}")
print(f"Sorted: {df_wrangled['date'].is_monotonic_increasing}")


Shape: (830, 26)
Date range: 2024-01-01 00:00:00 to 2026-04-17 00:00:00
Date dtype: datetime64[ns]
Sorted: True


# 4. Feature Engineering

Building on findings from the EDA notebook, this section engineers 
features for model training. Each transformation is documented with 
its physiological or statistical rationale.

Features are engineered in four categories:
1. Temporal features — capturing day-of-week and weekend effects
2. Lag features — yesterday's physiological state predicting today's recovery
3. Rolling averages — smoothing noise to capture trends
4. Log transforms — correcting skew identified in EDA

# 4.1 Temporal Features

In [ ]:
# Temporal features
# EDA identified strong day-of-week effect (Sunday avg 45.9 vs Monday 69.1)
# Month excluded — no meaningful seasonal pattern detected in EDA

df_features = df_wrangled.copy()
df_features['day_of_week'] = df_features['date'].dt.dayofweek

print(df_features[['date', 'day_of_week']].head(7))

# Important just to note that day of week is 0 to 6 index, Monday is 0 and Sunday is 6

        date  day_of_week
0 2024-01-01            0
1 2024-01-02            1
2 2024-01-03            2
3 2024-01-04            3
4 2024-01-05            4
5 2024-01-06            5
6 2024-01-07            6


# 4.2 Lag Features

No shifting of the data points is required as todays metrics such as, current day strain, recovery score, sleep duration, hrv & resting heart rate all will be aligned today to predict tomorrows score.

The one metric that can have an impact up to 48 hours is strain so we will take a column for todays strain and yesterdays strain.

The work needed here is to provided rolling 7 day averages to understand recent trends in recovery and strain (e.g. are we in a higher period of training which might drive down recovery score due to fatigue).

In [21]:
df_features['previous_day_strain'] = df_features['cycle_strain'].shift(1)

## Rolling 7 day averages

cols = ['hrv_rmssd_milli',
        'resting_heart_rate',
        'cycle_strain',
]

for col in cols:
    df_features[f'{col}_7d_avg'] = df_features[col].rolling(window=7, min_periods=3).mean()

df_features.describe().T

,count,mean,min,25%,50%,75%,max,std
cycle_id,830.0,860449880.626506,485128018.0,624350278.25,812161227.0,1073064303.5,1437064952.0,269179245.919201
date,830,2025-02-22 20:10:59.277108480,2024-01-01 00:00:00,2024-07-27 06:00:00,2025-02-23 12:00:00,2025-09-21 18:00:00,2026-04-17 00:00:00,NaN
recovery_score,830.0,60.304819,1.0,47.0,62.0,75.0,98.0,20.296294
hrv_rmssd_milli,830.0,66.328122,16.728266,58.351706,67.376565,75.552805,105.77171,14.097413
resting_heart_rate,830.0,59.415663,51.0,56.0,58.0,61.0,86.0,5.138093
spo2_percentage,828.0,96.283414,92.57143,95.712187,96.333336,96.941995,98.85714,0.989455
skin_temp_celsius,820.0,34.954435,33.105,34.757999,35.0,35.2,35.9,0.347853
respiratory_rate,830.0,13.522832,12.30957,13.242188,13.476562,13.710938,16.289062,0.443625
total_in_bed_time_hours,830.0,7.369268,3.103082,6.480587,7.241685,8.177651,11.984278,1.239028
total_awake_time_hours,830.0,0.675484,0.058333,0.456622,0.617515,0.823154,2.656838,0.326936
